In [1]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import zipfile
import os
import warnings
import chardet
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from google.cloud import bigquery
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [3]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "TRAMAS_BBVA_GCP/2026/Noviembre_26/"
#FOLDER_PATH= "TRAMAS_BBVA_GCP/TEMP/"
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_diarias_BBVA_2026"
#TABLE_ID= "temp_tramas_natalia"
TABLE_CONTROL_ID= "CONTROL_tramas_diarias"
#TABLE_CONTROL_ID= "CONTROL_tramas_temp"


In [4]:
# @title
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))

espacios_col = [(0, 3), (3, 23), (23, 43), (43,45), (45, 48), (48, 49), (368, 376), (376, 384), (384, 392), (346, 347), (639, 641), (411, 426)
                ]
column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda",
                "Tipo de Movimiento", "Fecha de Afiliacion", "Fecha de inicio del seguro", "Fecha fin del seguro",
                "Periodo de pago", "Plan de Seguro", "Prima"
                ]

schema_Trama = [
        bigquery.SchemaField("TIPO_DE_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_INTERNO_DEL_CANAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DE_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DE_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DE_AFILIACION", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_DE_INICIO_DEL_SEGURO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN_DEL_SEGURO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("PERIODO_DE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PLAN_DE_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TRAMA_ORIGINAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ERROR_PRIMA", bigquery.enums.SqlTypeNames.BOOL),
        bigquery.SchemaField("NOMBRE_DE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_DECLARADA", bigquery.enums.SqlTypeNames.DATETIME)
]

schema_Control = [
        bigquery.SchemaField("ARCHIVO_TXT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_TRAMA", bigquery.enums.SqlTypeNames.DATE)
]


# Extraer Fecha del nombre de archivo
def Extraer_fechas(filename):
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        # Primera fecha (ddmmyy → fecha completa)
        date_str = base[3:9]   # "030120"
        fecha_trama = datetime.strptime(date_str, "%d%m%y").date()

        # Segunda fecha (ddmm → usar año de la primera fecha)
        decl_str = base[10:14]  # "0601"
        dia, mes = int(decl_str[:2]), int(decl_str[2:])
        fecha_declarada = datetime(fecha_trama.year, mes, dia).date()
        if fecha_declarada < fecha_trama:
            fecha_declarada = fecha_declarada.replace(year=fecha_declarada.year + 1)
        return fecha_trama, fecha_declarada
    except Exception:
        return None, None  # Si algo falla


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan

        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo

def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise


def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0


#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref, schema=schema)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        print(f'ℹ️ ----- Se ha creado la tabla: {table_id} en el dataset: {dataset_id} -----')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return


def Procesar_File_txt(lista_archivos):
  for blob in lista_archivos:
    if blob.name.lower().endswith(".txt"):
      file_name = os.path.basename(blob.name)
      if not buscar_archivo(file_name):
          print(f"⏳ ... PROCESANDO ARCHIVO: {blob.name} ... ⏳")
          file_bytes = blob.download_as_bytes()
          try:
            contenido = file_bytes.decode('latin-1')
          except UnicodeDecodeError:
            # Si falla, usar UTF-8
            contenido = file_bytes.decode('utf-8', errors='replace')
          lista_lineas = contenido.splitlines()
          fecha_trama, fecha_declarada = Extraer_fechas(file_name)
          df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
          df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
          df_tramas["Error Prima"] = df_tramas["Prima Bruta"].isna()
          df_tramas["Fecha de Afiliacion"] = Convertir_fecha(df_tramas["Fecha de Afiliacion"])
          df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
          df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

          df_tramas["Nombre de Archivo"] = file_name
          df_tramas["Fecha Trama"] = fecha_trama
          df_tramas["Fecha Declarada"] = fecha_declarada
          df_tramas['Fecha Trama'] = pd.to_datetime(df_tramas['Fecha Trama'], errors='coerce')
          df_tramas['Fecha Declarada'] = pd.to_datetime(df_tramas['Fecha Declarada'], errors='coerce')
          df_tramas.columns = (df_tramas.columns.str.strip()  # quitar espacios al inicio/fin
                        .str.upper()  # opcional: todo en mayúsculas
                        .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                        )
          # Cambiar el nombre de tipo de seguro segun la condicion
          df_tramas.loc[(df_tramas['TIPO_DE_SEGURO'] == '803') & (df_tramas['PLAN_DE_SEGURO'] != '01') , 'TIPO_DE_SEGURO'] = '803_TLMK'

          # Guardar tabla en BigQuery
          fecha_carga = datetime.now()
          df_control = pd.DataFrame([{"ARCHIVO_TXT": file_name, "FECHA_CARGA": fecha_carga, "FECHA_TRAMA": fecha_trama}])
          Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID, schema_Trama)
          Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID, schema_Control)
          print(f"✅ ARCHIVO {file_name} CARGADO ...")
      else:
          print(f"⚠️ EL ARCHIVO {file_name} YA FUE CARGADO ANTERIORMENTE ...")
  return

##################### PRINCIPAL #########################
bucket = storage_client.bucket(BUCKET_NAME)
blob_list = list(bucket.list_blobs(prefix=FOLDER_PATH))
Procesar_File_txt(blob_list)


⚠️ EL ARCHIVO RED021126_0311.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED031126_0411.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED041126_0511.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED051126_0611.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED061126_0911.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED091126_1011.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED101126_1111.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED111126_1211.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED121126_1311.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED131126_1611.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED161126_1711.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED171126_1811.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED181126_1911.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED191126_2011.TXT YA FUE CARGADO ANTERIORMENTE ...
⚠️ EL ARCHIVO RED201126_2311.TXT YA FUE CARGADO ANTERIORMENTE ...
⏳ ... PROC

# Limpiar trama de caracteres extraños

In [5]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_diarias_BBVA_2026"
NOMBRE_TRAMA= 'RED231126_2411.TXT'

In [6]:
bigquery_client = bigquery.Client(project=PROJECT_ID)
def Ejecutar_Query(trama):
    tabla_tramas = bigquery_client.dataset(DATASET_ID).table(TABLE_ID)
    print(f"⏳ Limpiando Tramas del archivo '{NOMBRE_TRAMA}' ....")
    query = f"""
        SELECT TIPO_DE_SEGURO, TIPO_DE_MOVIMIENTO,
              REGEXP_REPLACE(TRAMA_ORIGINAL, '[^\\x20-\\x7E\\n\\r]|\\t', ' ') AS TRAMA_LIMPIA
        FROM `{tabla_tramas}`
        WHERE NOMBRE_DE_ARCHIVO = @archivo_txt
        ORDER BY TIPO_DE_SEGURO
        """
    job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", trama)])
    df = bigquery_client.query(query, job_config=job_config).to_dataframe()
    return df

df_trama= Ejecutar_Query(NOMBRE_TRAMA)
archivo_salida= f"/content/{NOMBRE_TRAMA.split(".")[0]} - limpia.TXT"

with open(archivo_salida, "w", encoding="utf-8") as f:
    for linea in df_trama["TRAMA_LIMPIA"]:
        f.write(str(linea)+ "\n")

print(f"✅ Archivo generado: {archivo_salida}")

⏳ Limpiando Tramas del archivo 'RED231126_2411.TXT' ....
✅ Archivo generado: /content/RED231126_2411 - limpia.TXT


##PRUEBAS

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
blob_list = list(bucket.list_blobs(prefix=FOLDER_PATH))
for blob in blob_list:
    if blob.name.endswith(".TXT"):
      print(f"⏳ ... PROCESANDO ARCHIVO: {blob.name} ... ⏳")

In [ ]:
resultado = chardet.detect(file_bytes)
print(resultado)

{'encoding': 'ISO-8859-1', 'confidence': 0.7299689922450754, 'language': ''}


In [ ]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102390 entries, 0 to 102389
Data columns (total 16 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   TIPO_DE_SEGURO              102390 non-null  object        
 1   CERTIFICADO                 102390 non-null  object        
 2   NUMERO_INTERNO_DEL_CANAL    102390 non-null  object        
 3   TIPO_DE_REGISTRO            102390 non-null  object        
 4   MONEDA                      102390 non-null  object        
 5   TIPO_DE_MOVIMIENTO          102390 non-null  object        
 6   FECHA_DE_AFILIACION         52169 non-null   datetime64[ns]
 7   FECHA_DE_INICIO_DEL_SEGURO  60332 non-null   datetime64[ns]
 8   FECHA_FIN_DEL_SEGURO        60406 non-null   datetime64[ns]
 9   PERIODO_DE_PAGO             102390 non-null  object        
 10  PRIMA                       102390 non-null  object        
 11  TRAMA_ORIGINAL              102390 non-

In [ ]:
# Extraer Fecha del nombre de archivo
def Extraer_fechas(filename):
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        # Primera fecha (ddmmyy → fecha completa)
        date_str = base[3:9]   # "030120"
        fecha_trama = datetime.strptime(date_str, "%d%m%y").date()

        # Segunda fecha (ddmm → usar año de la primera fecha)
        decl_str = base[10:14]  # "0601"
        dia, mes = int(decl_str[:2]), int(decl_str[2:])
        fecha_declarada = datetime(fecha_trama.year, mes, dia).date()

        return fecha_trama, fecha_declarada
    except Exception:
        return None, None  # Si algo falla


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan

        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo



#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    #print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return



def Procesar_File_txt(lista_tramas):
    for file in lista_tramas:
        file_content= archivo_zip.read(file)
        lista_lineas = file_content.decode('latin-1').splitlines()
        fecha_trama, fecha_declarada = Extraer_fechas(file)
        df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
        df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
        df_tramas["Error Prima"] = df_tramas["Prima Bruta"].isna()
        df_tramas["Fecha de Afiliacion"] = Convertir_fecha(df_tramas["Fecha de Afiliacion"])
        df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
        df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

        df_tramas["Fecha Trama"] = fecha_trama
        df_tramas["Fecha Declarada"] = fecha_declarada
        df_tramas['Fecha Trama'] = pd.to_datetime(df_tramas['Fecha Trama'], errors='coerce')
        df_tramas['Fecha Declarada'] = pd.to_datetime(df_tramas['Fecha Declarada'], errors='coerce')
        df_tramas.columns = (df_tramas.columns.str.strip()  # quitar espacios al inicio/fin
                      .str.upper()  # opcional: todo en mayúsculas
                      .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                      )

        # Guardar tabla en BigQuery
        Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID, schema_Trama)
        print(f"ARCHIVO {file} CARGADO...")
    return


CARGAR ARCHIVO TXT DESDE EL DRIVE

In [ ]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
from google.colab import drive
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
DATASET_ID = "produccion"
TABLE_ID= "tramas_sustento_sergio"

storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))
espacios_col = [(0, 3), (3, 23), (23, 43), (43,45), (45, 48), (48, 49), (376, 384), (384, 392), (346, 347), (396, 411), (411, 426)
                ]

column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda",
                "Tipo de Movimiento", "Fecha de inicio del seguro", "Fecha fin del seguro",
                "Periodo de pago", "Monto Asegurado", "Prima"
                ]

In [ ]:
with open("/content/drive/MyDrive/M_SUSTCRIS100_DESGRAVAMEN.txt", "r", encoding="latin-1") as file:
    lista_lineas =  file.readlines()


In [ ]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
df_tramas["Monto Asegurado"] = df_tramas["Monto Asegurado"].apply(convertir_prima)
df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

df_tramas.columns = (df_tramas.columns.str.strip()  # quitar espacios al inicio/fin
                      .str.upper()  # opcional: todo en mayúsculas
                      .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                      )

In [ ]:
df_tramas.head()

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,MONTO_ASEGURADO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA
0,901,00110002324000105283,00110057710219211849,01,PEN,4,2020-04-06,2020-05-06,M,0.0,000000000001400,901001100023240001052830011005771021921184901P...,14.0
1,901,00110002324000105283,00110057710219211849,01,PEN,4,2020-05-05,2020-06-05,M,0.0,000000000001400,901001100023240001052830011005771021921184901P...,14.0
2,901,00110002324000105283,00110057710219211849,01,PEN,4,2020-06-05,2020-07-05,M,0.0,000000000001400,901001100023240001052830011005771021921184901P...,14.0
3,901,00110002324000105283,00110057710219211849,01,PEN,4,2020-07-06,2020-08-06,M,0.0,000000000001400,901001100023240001052830011005771021921184901P...,14.0
4,901,00110002324000105283,00110057770252744798,01,PEN,4,2020-08-05,2020-09-05,M,0.0,000000000001400,901001100023240001052830011005777025274479801P...,14.0


In [ ]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4206 entries, 0 to 4205
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   TIPO_DE_SEGURO              4206 non-null   object        
 1   CERTIFICADO                 4206 non-null   object        
 2   NUMERO_INTERNO_DEL_CANAL    4206 non-null   object        
 3   TIPO_DE_REGISTRO            4206 non-null   object        
 4   MONEDA                      4206 non-null   object        
 5   TIPO_DE_MOVIMIENTO          4206 non-null   object        
 6   FECHA_DE_INICIO_DEL_SEGURO  4206 non-null   datetime64[ns]
 7   FECHA_FIN_DEL_SEGURO        4206 non-null   datetime64[ns]
 8   PERIODO_DE_PAGO             4206 non-null   object        
 9   MONTO_ASEGURADO             4206 non-null   float64       
 10  PRIMA                       4206 non-null   object        
 11  TRAMA_ORIGINAL              4206 non-null   object      

In [ ]:
schema_Trama = [
        bigquery.SchemaField("TIPO_DE_SEGURO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_INTERNO_DEL_CANAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DE_REGISTRO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DE_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_DE_INICIO_DEL_SEGURO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("FECHA_FIN_DEL_SEGURO", bigquery.enums.SqlTypeNames.DATETIME),
        bigquery.SchemaField("PERIODO_DE_PAGO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_ASEGURADO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TRAMA_ORIGINAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),

]

Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID, schema_Trama)

----- Se ha creado la tabla tramas_sustento_sergio en el dataset produccion -----
